# MuleNet-X Omega — Exploratory Analysis

Load a completed experiment run and explore the generated dataset.

Set `RUN_DIR` below to point at a run folder under `data/runs/`.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False})

# ── Configure this path ───────────────────────────────────────────────────
RUN_DIR = "../data/runs/"   # update with a specific run_id subfolder
# ──────────────────────────────────────────────────────────────────────────

# Auto-select the most recent run if RUN_DIR points to the runs/ parent
if not os.path.exists(os.path.join(RUN_DIR, 'accounts.csv')):
    runs = sorted(os.listdir(RUN_DIR))
    if runs:
        RUN_DIR = os.path.join(RUN_DIR, runs[-1])
        print(f'Auto-selected run: {RUN_DIR}')

accounts_df     = pd.read_csv(os.path.join(RUN_DIR, 'accounts.csv'))
transactions_df = pd.read_csv(os.path.join(RUN_DIR, 'transactions.csv'))
features_df     = pd.read_csv(os.path.join(RUN_DIR, 'features.csv'))
risk_df         = pd.read_csv(os.path.join(RUN_DIR, 'risk_scores.csv'))

print(f'Accounts    : {len(accounts_df):,}')
print(f'Transactions: {len(transactions_df):,}')
accounts_df.head()

## 1 · Degree Distribution

Compares the in-degree and out-degree distributions across account types.
Mule accounts typically show elevated out-degree relative to their in-degree.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, col, title in zip(axes, ['in_degree', 'out_degree'], ['In-Degree', 'Out-Degree']):
    for atype, grp in features_df.groupby('account_type'):
        ax.hist(grp[col], bins=30, alpha=0.65, label=atype, edgecolor='none')
    ax.set_title(f'{title} Distribution by Account Type')
    ax.set_xlabel(title)
    ax.set_ylabel('Count')
    ax.legend()

plt.tight_layout()
plt.savefig(os.path.join(RUN_DIR, 'plot_degree_distribution.png'), bbox_inches='tight')
plt.show()

## 2 · Fraud vs Normal Transaction Counts

Shows the absolute and proportional split between labelled normal and
mule-flow transactions.

In [ ]:
label_counts = transactions_df['label'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Bar chart — absolute counts
colors = ['#4C72B0', '#DD8452']
axes[0].bar(label_counts.index, label_counts.values, color=colors, edgecolor='white')
axes[0].set_title('Transaction Count by Label')
axes[0].set_ylabel('Count')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
for i, v in enumerate(label_counts.values):
    axes[0].text(i, v + label_counts.max() * 0.01, f'{v:,}', ha='center', fontsize=9)

# Pie chart — proportions
axes[1].pie(label_counts.values, labels=label_counts.index, autopct='%1.1f%%',
            colors=colors, startangle=140, wedgeprops={'edgecolor': 'white'})
axes[1].set_title('Label Proportion')

plt.tight_layout()
plt.savefig(os.path.join(RUN_DIR, 'plot_label_distribution.png'), bbox_inches='tight')
plt.show()

## 3 · Risk Score Distribution

Plots the baseline risk score histogram split by ground-truth label.
A well-separating scorer should show mule accounts skewing toward higher scores.

In [ ]:
merged = features_df[['account_id', 'is_mule']].merge(risk_df, on='account_id')

fig, ax = plt.subplots(figsize=(10, 4))

for is_mule, label, color in [(False, 'Normal', '#4C72B0'), (True, 'Mule', '#DD8452')]:
    subset = merged[merged['is_mule'] == is_mule]['risk_score']
    ax.hist(subset, bins=40, alpha=0.70, label=f'{label} (n={len(subset):,})',
            color=color, edgecolor='none')

ax.axvline(0.5, color='red', linestyle='--', linewidth=1.2, label='threshold=0.5')
ax.set_title('Risk Score Distribution — Mule vs Normal')
ax.set_xlabel('Risk Score')
ax.set_ylabel('Count')
ax.legend()

plt.tight_layout()
plt.savefig(os.path.join(RUN_DIR, 'plot_risk_distribution.png'), bbox_inches='tight')
plt.show()

print('\nRisk score summary by group:')
print(merged.groupby('is_mule')['risk_score'].describe().round(4).to_string())

## 4 · Feature Correlation Heatmap

Shows pairwise Pearson correlation among the six engineered features.
Strongly correlated features may be redundant for downstream modelling.

In [ ]:
import numpy as np

feat_cols = ['in_degree', 'out_degree', 'total_in', 'total_out',
             'velocity_score', 'imbalance_score']
corr = features_df[feat_cols].corr()

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(corr, cmap='RdYlBu_r', vmin=-1, vmax=1)
plt.colorbar(im, ax=ax, shrink=0.8)
ax.set_xticks(range(len(feat_cols)))
ax.set_yticks(range(len(feat_cols)))
ax.set_xticklabels(feat_cols, rotation=35, ha='right', fontsize=9)
ax.set_yticklabels(feat_cols, fontsize=9)
for i in range(len(feat_cols)):
    for j in range(len(feat_cols)):
        ax.text(j, i, f'{corr.iloc[i, j]:.2f}', ha='center', va='center', fontsize=8)
ax.set_title('Feature Correlation Matrix')
plt.tight_layout()
plt.savefig(os.path.join(RUN_DIR, 'plot_feature_correlation.png'), bbox_inches='tight')
plt.show()